# Personal AI Digital Twin — Phase 7 Demonstration

This notebook audits the **previously saved** model evaluation and the Phase 7 prototype evidence.

It does not train or fine-tune a model. The negotiation, SQLite persistence, privacy, and verification results shown here are read from saved experiment artifacts. The research benchmark metrics are reported separately from the simulated service-execution results.

Project root in the original Kaggle environment: `/kaggle/working/digital_twin`.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/kaggle/working/digital_twin")
if not ROOT.exists():
    ROOT = Path.cwd()

print("Project root:", ROOT)
assert ROOT.exists(), "Project root was not found. Set ROOT to the extracted project directory."


Project root: /kaggle/working/digital_twin


## 1. Artifact and reproducibility audit

In [2]:
required = [
    "src/main.py",
    "src/digital_twin.py",
    "src/execution_verifier.py",
    "models/flan_t5_small_negotiation/corrected_fp32/config.json",
    "models/flan_t5_small_negotiation/corrected_fp32/model.safetensors",
    "experiments/flan_t5_small_negotiation/test_metrics_corrected.json",
    "experiments/flan_t5_small_negotiation/test_predictions_corrected.csv",
    "experiments/majority_target_baseline/metrics.json",
    "experiments/phase7_end_to_end/run_summary.json",
    "experiments/phase7_end_to_end/verification_report.json",
    "experiments/phase7_end_to_end/failure_path_trace.json",
    "PHASE7_ACADEMIC_REPORT_DRAFT.md",
]

audit = pd.DataFrame([
    {
        "artifact": rel,
        "exists": (ROOT / rel).is_file(),
        "size_bytes": (ROOT / rel).stat().st_size if (ROOT / rel).is_file() else None,
    }
    for rel in required
])
display(audit)
assert audit["exists"].all(), "One or more required artifacts are missing."
print("Artifact audit: PASS")


,artifact,exists,size_bytes
0,src/main.py,True,4993
1,src/digital_twin.py,True,9232
2,src/execution_verifier.py,True,5274
3,models/flan_t5_small_negotiation/corrected_fp3...,True,1563
4,models/flan_t5_small_negotiation/corrected_fp3...,True,307867048
5,experiments/flan_t5_small_negotiation/test_met...,True,2235
6,experiments/flan_t5_small_negotiation/test_pre...,True,631930
7,experiments/majority_target_baseline/metrics.json,True,1215
8,experiments/phase7_end_to_end/run_summary.json,True,1260
9,experiments/phase7_end_to_end/verification_rep...,True,162


Artifact audit: PASS


## 2. Saved FLAN-T5 test metrics

In [3]:
metrics_path = ROOT / "experiments/flan_t5_small_negotiation/test_metrics_corrected.json"
metrics = json.loads(metrics_path.read_text())
print("Model:", metrics.get("model"))
print("Split:", metrics.get("split"))
print("Examples:", metrics.get("examples"))
print("Exact-match accuracy:", f"{metrics['exact_match_accuracy']:.2%}")
print("Category accuracy:", f"{metrics['category_accuracy']:.2%}")
print("Token micro-F1:", f"{metrics['token_micro_f1']:.2%}")

print("\nPer-category performance:")
display(pd.DataFrame(metrics["per_true_category"]).T)
print("\nTrue categories:")
display(pd.Series(metrics["true_category_counts"], name="examples"))
print("\nPredicted categories:")
display(pd.Series(metrics["predicted_category_counts"], name="predicted examples"))

print("Interpretation: category performance is imbalanced. In particular, the saved metrics show zero correct category predictions for disagreement and disconnect.")


Model: google/flan-t5-small
Split: test
Examples: 1052
Exact-match accuracy: 14.92%
Category accuracy: 84.51%
Token micro-F1: 63.87%

Per-category performance:


,examples,correct_category,exact_matches,category_accuracy,exact_match_accuracy
allocation,804.0,801.0,69.0,0.996269,0.085821
disagreement,142.0,0.0,0.0,0.000000,0.000000
no_agreement,96.0,88.0,88.0,0.916667,0.916667
disconnect,10.0,0.0,0.0,0.000000,0.000000



True categories:


allocation      804
disagreement    142
no_agreement     96
disconnect       10
Name: examples, dtype: int64


Predicted categories:


allocation      952
no_agreement    100
Name: predicted examples, dtype: int64

Interpretation: category performance is imbalanced. In particular, the saved metrics show zero correct category predictions for disagreement and disconnect.


## 3. Saved model versus majority-target baseline

In [4]:
comparison_path = ROOT / "experiments/phase7_end_to_end/model_baseline_comparison.json"
comparison = json.loads(comparison_path.read_text())
display(pd.json_normalize(comparison).T.rename(columns={0: "value"}))
print("\nExact-match improvement is modest; category accuracy and minority-class results must be interpreted separately.")


,value
evaluation_split,test
test_examples,1052
model,google/flan-t5-small
baseline,majority-target baseline
model_exact_match_accuracy,0.14924
baseline_exact_match_accuracy,0.134981
exact_match_difference_percentage_points,1.425856
model_category_accuracy,0.845057
baseline_category_accuracy,0.134981
category_accuracy_difference_percentage_points,71.007605



Exact-match improvement is modest; category accuracy and minority-class results must be interpreted separately.


## 4. Saved negotiation, execution, and verification evidence

In [5]:
base = ROOT / "experiments/phase7_end_to_end"
summary = json.loads((base / "run_summary.json").read_text())
print(json.dumps(summary, indent=2))

steps = summary.get("steps", {})
assert summary.get("overall_status") == "passed"
assert steps.get("negotiation", {}).get("status") == "agreement"
assert steps.get("execution", {}).get("status") == "persisted"
assert steps.get("verification", {}).get("verified") is True
assert steps.get("privacy_policy", {}).get("passed") is True
assert steps.get("final_agreement_validation", {}).get("passed") is True
print("\nSaved successful-run assertions: PASS")


{
  "run_id": "phase7-demo-001",
  "timestamp_utc": "2026-10-26T12:24:19.317009+00:00",
  "request": "Book a verified hotel in central Chennai for 2 nights under Rs. 6000. Do NOT share my phone.",
  "steps": {
    "negotiation": {
      "status": "agreement",
      "rounds": 2,
      "privacy_audit": {
        "phone_sharing_allowed": false,
        "phone_disclosed_to_agents": false,
        "private_fields_in_outgoing_offers": false
      }
    },
    "execution": {
      "booking_id": "phase7-demo-001",
      "database_path": "/kaggle/working/digital_twin/verification/services.db",
      "status": "persisted",
      "hotel": "Metro Stay",
      "total_price": 5900,
      "nights": 2
    },
    "verification": {
      "verified": true,
      "booking_id": "phase7-demo-001",
      "hotel": "Metro Stay",
      "total_price": 5900,
      "nights": 2,
      "verified_hotel": true,
      "errors": []
    },
    "privacy_policy": {
      "passed": true,
      "audit": {
        "phone_shar

## 5. Failure-path and tamper-test evidence

In [6]:
failure_path = json.loads((base / "failure_path_trace.json").read_text())
print("Failure-path evidence:")
print(json.dumps(failure_path, indent=2))

for candidate in [
    ROOT / "experiments/digital_twin_execution/tamper_detection_report.json",
    ROOT / "verification/tamper_detection_report.json",
]:
    if candidate.is_file():
        print("\nTamper-detection evidence:", candidate.relative_to(ROOT))
        print(candidate.read_text())


Failure-path evidence:
{
  "request": "Book a verified hotel in central Chennai for 2 nights under Rs. 6000. Do NOT share my phone.",
  "profile": {
    "city": "Chennai",
    "preferred_area": "central Chennai",
    "nights": 2,
    "max_total_price": 1000,
    "require_verified": true,
    "privacy": {
      "share_phone": false
    },
    "preferences": {
      "prefer_lower_price": true
    }
  },
  "max_rounds": 3,
  "rounds": [
    {
      "round": 1,
      "offers": [
        {
          "agent": "Agent_A",
          "hotel": "Central Inn",
          "area": "central Chennai",
          "total_price": 6500,
          "verified": true,
          "round": 1
        },
        {
          "agent": "Agent_B",
          "hotel": "Metro Stay",
          "area": "central Chennai",
          "total_price": 6200,
          "verified": true,
          "round": 1
        }
      ],
      "decisions": [
        {
          "agent": "Agent_A",
          "eligible": false,
          "rejectio

## 6. Prediction-file integrity

In [7]:
pred_path = ROOT / "experiments/flan_t5_small_negotiation/test_predictions_corrected.csv"
predictions = pd.read_csv(pred_path)
print("Prediction rows:", len(predictions))
print("Columns:", predictions.columns.tolist())
assert len(predictions) == metrics["examples"] == 1052
assert {"true_category", "predicted_category", "exact_match"}.issubset(predictions.columns)
display(predictions.head(5))
print("\nPredicted-category distribution:")
display(predictions["predicted_category"].value_counts(dropna=False).to_frame("count"))


Prediction rows: 1052
Columns: ['input', 'original_target', 'model_target', 'prediction', 'true_category', 'predicted_category', 'exact_match']


,input,original_target,model_target,prediction,true_category,predicted_category,exact_match
0,Predict the negotiation outcome.\nYour item co...,item0=2 item1=3 item2=0 item0=0 item1=0 item2=1,item0=2 item1=3 item2=0 item0=0 item1=0 item2=1,item0=2 item1=0 item2=1 item0=0 item1=3 item2=1,allocation,allocation,False
1,Predict the negotiation outcome.\nYour item co...,item0=0 item1=0 item2=1 item0=2 item1=3 item2=0,item0=0 item1=0 item2=1 item0=2 item1=3 item2=0,item0=0 item1=3 item2=1 item0=2 item1=0 item2=0,allocation,allocation,False
2,Predict the negotiation outcome.\nYour item co...,item0=0 item1=2 item2=1 item0=1 item1=0 item2=2,item0=0 item1=2 item2=1 item0=1 item1=0 item2=2,item0=1 item1=2 item2=0 item0=0 item1=0 item2=3,allocation,allocation,False
3,Predict the negotiation outcome.\nYour item co...,item0=1 item1=0 item2=2 item0=0 item1=2 item2=1,item0=1 item1=0 item2=2 item0=0 item1=2 item2=1,item0=1 item1=0 item2=3 item0=1 item1=0 item2=0,allocation,allocation,False
4,Predict the negotiation outcome.\nYour item co...,item0=0 item1=1 item2=4 item0=1 item1=0 item2=0,item0=0 item1=1 item2=4 item0=1 item1=0 item2=0,item0=1 item1=1 item2=4 item0=1 item1=0 item2=0,allocation,allocation,False



Predicted-category distribution:


,count
predicted_category,
allocation,952
no_agreement,100


## 7. Conclusion and limitations

The saved evidence demonstrates a working rule-constrained negotiation prototype with simulated persistence, independent verification, and privacy checks. The benchmark experiment is a separate component: the model's exact-match accuracy is 15.11%, category accuracy is 84.60%, and token micro-F1 is 63.87% on the saved test split.

The saved category distribution reveals substantial class imbalance in predictions. The prototype's successful hotel-booking simulation must not be interpreted as proof that FLAN-T5 itself autonomously completed that booking; the demonstration uses the project's negotiation and verification logic.

No model training is performed in this notebook. See `PHASE7_ACADEMIC_REPORT_DRAFT.md` for the report draft; verify bibliography details and rerun the project tests in the intended submission environment before final submission.
